In [3]:
#cài đặt thư viện kagglehub để lấy dataset
%pip install kagglehub
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark delta-spark


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00


In [4]:
import kagglehub
import os
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
builder = SparkSession.builder.appName("AmazonProducts") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
#Tải dataset
path = kagglehub.dataset_download("lokeshparab/amazon-products-dataset")

#Tìm TẤT CẢ file CSV
csv_files = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

print(f"Found total: {len(csv_files)} files. Merging files altogether...")

#Định nghĩa Schema
schema = StructType([
    StructField("name", StringType(), True),
    StructField("main_category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("image", StringType(), True),
    StructField("link", StringType(), True),
    StructField("ratings", StringType(), True),
    StructField("no_of_ratings", StringType(), True),
    StructField("discount_price", StringType(), True),
    StructField("actual_price", StringType(), True)
])

pdf_list = []
for i, file_path in enumerate(csv_files):
    try:
        temp_pdf = pd.read_csv(file_path, dtype=str, quotechar='"', on_bad_lines='skip', low_memory=False)

        for col in ["name", "main_category", "sub_category", "image", "link", "ratings", "no_of_ratings", "discount_price", "actual_price"]:
            if col not in temp_pdf.columns:
                temp_pdf[col] = ""

        temp_pdf = temp_pdf[["name", "main_category", "sub_category", "image", "link", "ratings", "no_of_ratings", "discount_price", "actual_price"]]

        pdf_list.append(temp_pdf)
        if (i+1) % 20 == 0: print(f"Đã xử lý {i+1}/{len(csv_files)} files...")
    except Exception as e:
        print(f"Ignore errror file {file_path}: {e}")

# Hợp nhất thành 1 Pandas DF duy nhất
full_pdf = pd.concat(pdf_list, ignore_index=True).fillna("").astype(str)

spark.sql("DROP TABLE IF EXISTS default.amazon_bronze")
df_bronze = spark.createDataFrame(full_pdf, schema=schema)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("default.amazon_bronze")

print(f"Total number of rows of dataset: {df_bronze.count()}")
display(df_bronze.limit(10))

file_name = "amazon_products_merged.csv"
full_pdf.to_csv(file_name, index = False, encoding = 'utf-8-sig')




Using Colab cache for faster access to the 'amazon-products-dataset' dataset.
Found total: 140 files. Merging files altogether...
Đã xử lý 20/140 files...
Đã xử lý 40/140 files...
Đã xử lý 60/140 files...
Đã xử lý 80/140 files...
Đã xử lý 100/140 files...
Đã xử lý 120/140 files...
Đã xử lý 140/140 files...
Total number of rows of dataset: 1103170


DataFrame[name: string, main_category: string, sub_category: string, image: string, link: string, ratings: string, no_of_ratings: string, discount_price: string, actual_price: string]

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
price_fallback = 1499.0
rating_fallback = 4.0
price_min, price_max = 0, 1000000
rating_min, rating_max = 1.0, 5.0
df_bronze = spark.table('default.amazon_bronze')
dedup_window = Window.partitionBy("name").orderBy(
    F.desc(F.length(F.col("actual_price"))),
    F.desc(F.length(F.col("ratings"))),
    F.col("name")
)
df_silver = df_bronze \
    .filter(F.col('name').isNotNull() & (F.col('name') != "")) \
    .withColumn("_rn", F.row_number().over(dedup_window)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn") \
    .withColumn(
        'ratings_raw',
        F.expr("try_cast(regexp_extract(ratings, '([0-9]\\.[0-9])', 1) as float)")
    ) \
    .withColumn(
        'ratings_final',
        F.when(
            (F.col("ratings_raw") >= 1.0) & (F.col("ratings_raw") <= 5.0),
            F.col("ratings_raw").cast("decimal(2,1)")
        ).otherwise(F.lit(rating_fallback).cast("decimal(2,1)"))
    ) \
    .withColumn(
        'price_raw',
        F.expr("try_cast(regexp_replace(actual_price, '[^0-9.]', '') as float)")
    ) \
    .withColumn(
        'price_final',
        F.when((F.col("price_raw") > 0) & (F.col("price_raw") < 1000000), F.col("price_raw"))
         .otherwise(F.lit(price_fallback))
    ) \
    .withColumn(
        'no_of_ratings',
        F.coalesce(
            F.expr("try_cast(regexp_replace(no_of_ratings, '[^0-9]', '') as int)"),
            F.lit(0)
        )
    ) \
    .withColumn(
        'price_tier',
        F.when(F.col('price_raw') < 500, 'budget')
         .when(F.col('price_raw') < 5000, 'mid-range')
         .when(F.col('price_raw') >= 5000, 'premium')
         .otherwise('unknown price')
    ) \
    .withColumn(
        'rating_tier',
        F.when(F.col('ratings_final') >= 4.5, 'top rated')
         .when(F.col('ratings_final') >= 3.5, 'well rated')
         .otherwise('average rated')
    ) \
    .withColumn(
        'combined_text',
        F.trim(F.concat_ws(" | ",
            F.col("name"),
            F.coalesce(F.col("main_category"), F.lit("unknown")),
            F.coalesce(F.col("sub_category"), F.lit("unknown")),
            F.col('price_tier'),
            F.col('rating_tier')
        ))
    ) \
    .select("name", "main_category", "sub_category", "image", "link",
            "ratings_final", "no_of_ratings", "price_final", "combined_text")

spark.sql('DROP TABLE IF EXISTS default.amazon_silver')
df_silver.write.format('delta').mode('overwrite').saveAsTable('default.amazon_silver')
print(f'SILVER DONE')
print(f"Number of rows after cleaning: {df_silver.count()}")
display(df_silver.limit(10))

SILVER DONE
Number of rows after cleaning: 396210


DataFrame[name: string, main_category: string, sub_category: string, image: string, link: string, ratings_final: decimal(2,1), no_of_ratings: int, price_final: double, combined_text: string]

In [7]:
# Null/ Missing Value Audit

from pyspark.sql import functions as F
df_silver = spark.table('default.amazon_silver')
total_rows = df_silver.count()
print(f'Total rows: {total_rows:,}')
print('=' * 55)

null_check = []
for col_name in df_silver.columns:
    null_count = df_silver.filter(
        F.col(col_name).isNull() | (F.col(col_name).cast("string") == "")
    ).count()
    null_check.append({
        'column': col_name,
        'null_count': null_count,
        'null_pct': round(null_count / total_rows * 100, 2)
    })
null_df = spark.createDataFrame(null_check).orderBy(F.desc("null_count"))
display(null_df.toPandas())

Total rows: 396,210


,column,null_count,null_pct
0,name,0,0.0
1,main_category,0,0.0
2,sub_category,0,0.0
3,image,0,0.0
4,link,0,0.0
5,ratings_final,0,0.0
6,no_of_ratings,0,0.0
7,price_final,0,0.0
8,combined_text,0,0.0


In [ ]:
#Exploratory Data Analysis
#Categorical Analysis
df_silver = spark.table('default.amazon_silver')
print('---TOP 10 MAIN CATEGORIES')
category_stats = df_silver.groupBy('main_category').count().orderBy(F.desc('count'))
display(category_stats.limit(10))

print('---TOP 10 SUB CATEGORIES')
sub_category_stats = df_silver.groupBy('sub_category').count().orderBy(F.desc('count'))
display(sub_category_stats.limit(10))

#Numerical Analysis
print('DESCRIPTIVE ANALYSIS (RATINGS & PRICE)')
display(df_silver.select('ratings_final', 'price_final').summary())

#Ratings Distribution
print('RATINGS DISTRIBUTION')
display(df_silver.groupBy('ratings_final').count().orderBy('ratings_final'))

#Price Bucketing
print('PRICE BUCKETING')
df_price_bucket = df_silver.withColumn("price_range",
    F.when(F.col("price_final") < 500, "< 500")
     .when((F.col("price_final") >= 500) & (F.col("price_final") < 2000), "500 - 2000")
     .when((F.col("price_final") >= 2000) & (F.col("price_final") < 10000), "2000 - 10000")
     .otherwise("> 10000")
)
display(df_price_bucket.groupBy('price_range').count())

#Text Length
df_text_stats = df_silver.withColumn('text_length', F.length('combined_text'))
print('TEXT LENGTH')
display(df_text_stats.select('text_length').summary())

print('TEXT WITH LESS THAN 20 ITEMS')
display(df_text_stats.filter("text_length < 20").select("name", "combined_text").limit(5))

In [ ]:
#Correlation Analysis
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols = ['ratings_final', 'price_final'], outputCol= 'features')
df_corr = spark.table('default.amazon_silver') \
    .withColumn('ratings_final', F.col('ratings_final').cast('float')) \
    .withColumn('price_final', F.col('price_final').cast('float'))

df_vector = assembler.transform(df_corr)

#Pearson correlation matrix
matrix = Correlation.corr(df_vector, 'features').collect()[0][0]
cor_matrix = matrix.toArray().tolist()

corr = cor_matrix[0][1]
strength = 'weak' if abs(corr) < 0.3 else 'moderate' if abs(corr) < 0.6 else 'strong'
direction = 'positive' if corr > 0 else 'negative'
print(f'Correlation: {corr:.4f} → {strength} {direction} correlation')
print('Interpretation: Higher price does NOT guarantee higher rating' if abs(corr) < 0.3 else '')

In [ ]:
spark.table("default.amazon_silver").printSchema()

In [ ]:
#Gold Layer
%pip install sentence-transformers

In [ ]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, FloatType
from pyspark.sql.window import Window

_model_holder = None
@pandas_udf(ArrayType(FloatType()))
def compute_embeddings_safe(texts: pd.Series) -> pd.Series:
    global _model_holder
    if _model_holder is None:
        os.environ['HF_HOME'] = '/tmp/huggingface'
        os.environ['SENTENCE_TRANSFORMERS_HOME'] = '/tmp/sentence_transformers'
        _model_holder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder='/tmp/models')
    embeddings = _model_holder.encode(
        texts.tolist(),
        batch_size = 16,
        show_progress_bar = False,
        convert_to_numpy = True
    )
    return pd.Series(embeddings.tolist())

MAX_PER_CATEGORY = 500
ROWS_PER_PARTITION = 50
num_partitions = max(1, min(600, 30000 // ROWS_PER_PARTITION))

windowSpec = Window.partitionBy("main_category").orderBy(
    F.desc(F.col("ratings_final") * F.log1p(F.col("no_of_ratings").cast("float")))
)
df_top_per_cat = spark.table("default.amazon_silver") \
    .withColumn("rank", F.row_number().over(windowSpec)) \
    .filter(F.col("rank") <= MAX_PER_CATEGORY) \
    .drop("rank")

df_random = spark.table("default.amazon_silver").sample(False, 0.05, seed=42)

gold_dedup_window = Window.partitionBy("name").orderBy(
    F.desc(F.col("ratings_final") * F.log1p(F.col("no_of_ratings").cast("float")))
)
df_gold_input = df_top_per_cat.unionByName(df_random, allowMissingColumns=True) \
    .withColumn("_rn", F.row_number().over(gold_dedup_window)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn") \
    .orderBy(F.col("name")) \
    .limit(30000) \
    .repartition(num_partitions)

df_gold = df_gold_input.withColumn("vector", compute_embeddings_safe(F.col("combined_text")))

spark.sql("DROP TABLE IF EXISTS default.amazon_gold")
df_gold.write.format("delta").mode("overwrite").saveAsTable("default.amazon_gold")

print('--- GOLD LAYER HAS BEEN EXECUTED ---')

df_result = spark.table("default.amazon_gold")
display(df_result.groupBy("main_category").count().orderBy(F.desc("count")))

In [ ]:
df_check = spark.table("default.amazon_gold")
print(f"Total rows: {df_check.count():,}")
print(f"Columns: {df_check.columns}")
display(df_check.groupBy("main_category").count().orderBy(F.desc("count")))


In [ ]:
#Search Engine
%pip install faiss-cpu

import faiss
import pandas as pd
import numpy as np
import os
import time
from sentence_transformers import SentenceTransformer
from typing import Tuple, Dict

class ProductSearchEngine:
    def __init__(self):
        self.model = None
        self.index = None
        self.data = None
        self._initialized = False
        self._index_path = '/tmp/faiss_product_index.bin'
    def initialize(self, data_table: str = 'default.amazon_gold'):
        if self._initialized:
            return
        os.environ['HF_HOME'] = '/tmp/huggingface'
        self.model = SentenceTransformer('all-MiniLM-L6-v2', cache_folder='/tmp/models')
        self.data = spark.table(data_table).toPandas()
        vectors = np.array(self.data['vector'].tolist()).astype('float32')
        faiss.normalize_L2(vectors)
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)
        self._initialized = True
        print(f'Search Engine Initialized: {self.index.ntotal} products')
    def save_index(self, path: str = None):
        if not self._initialized:
            raise RuntimeError('Search engine is not initialized')
        save_path = path or self._index_path
        try:
            faiss.write_index(self.index, save_path)
            print(f'FAISS index save to: {save_path}')
            return save_path
        except Exception as e:
            print(f'Could not save index {e}')
            return None
    def load_index(self, path: str = None):
        load_path = path or self._index_path
        try:
            self.index = faiss.read_index(load_path)
            print(f' FAISS index loaded from: {load_path}')
            print(f' Vectors: {self.index.ntotal}, Dimensions: {self.index.d}')
            return True
        except Exception as e:
            print(f'Could not load index from {load_path}: {e}')
            return False
    def search(self, query:str, k: int = 50):
        if not self._initialized:
            raise RuntimeError('Search engine not initialized. Call initialize() first')
        start = time.time()
        #encode query
        q_vec = self.model.encode([query]).astype('float32')
        q_vec = np.ascontiguousarray(q_vec)
        faiss.normalize_L2(q_vec)
        #FAISS search
        scores, indices = self.index.search(q_vec, k)

        results = self.data.iloc[indices[0]].copy()
        results['similarity'] = scores[0]
        results = results.drop(columns= ['vector'], errors = 'ignore')
        total_latency = (time.time()- start) * 1000
        metadata = {
            'query': query,
            'latency_ms': total_latency,
            'num_results': len(results),
            'index_size': self.index.ntotal
        }
        return results, metadata

# Check if already initialized (fixed logic)
try:
    if search_engine._initialized:
        print('Search engine already initialized. Skipping')
    else:
        raise AttributeError('Not initialized')
except (NameError, AttributeError):
    print('Initializing search engine...')
    search_engine = ProductSearchEngine()
    search_engine.initialize()
    try:
        search_engine.save_index()
    except Exception as e:
        print(f'Could not persist index: {e}')
    print('\n✅ Search engine ready!')
    print(f'{search_engine.index.ntotal:,} products indexed')
    print(f'Use: results, meta = search_engine.search(query)')

**Model Evaluation & Comparison**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict

# Guard check: ensure search engine is ready
if 'search_engine' not in globals() or not search_engine._initialized:
    raise RuntimeError("❌ Search engine not initialized. Please run Cell 11 first.")

print("✅ Search engine detected and ready")

# Test Cases (FIXED: removed duplicate, fixed circular keywords)
test_cases = [
    {
        "query": "affordable wireless audio",
        "expected_keywords": ["earphone", "headphone", "earbud", "bluetooth", "tws"],
        "category": "Audio",
        "note": "Synonym test: 'affordable'='cheap', 'audio'='earphones/headphones'"
    },
    {
        "query": "portable computer for gaming",
        "expected_keywords": ["laptop", "notebook", "rog", "msi", "alienware"],
        "category": "Computing",
        "note": "Concept test: 'portable computer' should match gaming laptops"
    },
    {
        "query": "exercise equipment for floor workouts",
        "expected_keywords": ["mat", "yoga", "fitness", "gym"],
        "category": "Sports",
        "note": "Inference test: 'floor workouts' → yoga/exercise mats"
    },
    {
        "query": "mobile device with photography features",
        "expected_keywords": ["phone", "smartphone", "camera", "iphone", "pixel", "samsung"],
        "category": "Mobile",
        "note": "Descriptive test: 'mobile device'='smartphone', 'photography'='camera'"
    },
    {
        "query": "hydration container for athletes",
        "expected_keywords": ["bottle", "flask", "sport", "water", "sipper"],
        "category": "Sports",
        "note": "Functional test: 'hydration container'='water bottle'"
    },
    {
        "query": "footwear for jogging",
        "expected_keywords": ["shoe", "running", "sneaker", "sport", "athletic"],
        "category": "Footwear",
        "note": "Generic-to-specific test: 'footwear'='shoes', 'jogging'='running'"
    },
    {
        "query": "budget laptop for college",
        "expected_keywords": ["laptop", "notebook", "student", "lenovo", "hp", "dell"],
        "category": "Computing",
        "note": "Context test: 'budget'='affordable', 'college'='student'"
    },
    {
        "query": "cordless pointing device",
        "expected_keywords": ["mouse", "wireless", "bluetooth", "logitech"],
        "category": "Accessories",
        "note": "Technical synonym test: 'cordless'='wireless', 'pointing device'='mouse'"
    },
    {
        "query": "portable music player",
        "expected_keywords": ["ipod", "mp3", "walkman", "earphone"],
        "category": "Electronics",
        "note": "FIXED: Removed circular keywords 'music' and 'player'"
    },
    {
        "query": "timepiece for wrist",
        "expected_keywords": ["watch", "smartwatch", "fitness", "band"],
        "category": "Accessories",
        "note": "Formal description test: 'timepiece'='watch'"
    }
]

print(f"📋 Loaded {len(test_cases)} test queries across {len(set(tc['category'] for tc in test_cases))} categories")

# Keyword Search Baseline
def keyword_search(query_text: str, k: int = 10) -> pd.DataFrame:
    """Baseline: simple keyword matching"""
    keywords = query_text.lower().split()
    pdf = search_engine.data.copy()
    pdf['kw_score'] = pdf['name'].apply(
        lambda x: sum(1 for kw in keywords if kw in str(x).lower())
    )
    pdf = pdf.sort_values(['kw_score', 'ratings_final'], ascending=[False, False])
    return pdf.head(k)[['name', 'ratings_final', 'price_final', 'kw_score']]

# Evaluation Metrics
def precision_at_k(results_df: pd.DataFrame, expected_keywords: List[str], k: int = 5) -> float:
    """Calculate Precision@K"""
    top_k = results_df.head(k)
    hits = sum(
        1 for _, row in top_k.iterrows()
        if any(kw in str(row['name']).lower() for kw in expected_keywords)
    )
    return hits / k

def mean_reciprocal_rank(results_df: pd.DataFrame, expected_keywords: List[str]) -> float:
    """Calculate Mean Reciprocal Rank (MRR)"""
    for rank, (_, row) in enumerate(results_df.iterrows(), start=1):
        if any(kw in str(row['name']).lower() for kw in expected_keywords):
            return 1.0 / rank
    return 0.0

def evaluate_search_method(search_fn, test_cases: List[Dict], k: int = 5) -> pd.DataFrame:
    """Evaluate a search method across test cases"""
    results = []
    for case in test_cases:
        query = case['query']
        expected = case['expected_keywords']
        category = case['category']

        search_results = search_fn(query)

        p_at_k = precision_at_k(search_results, expected, k=k)
        mrr = mean_reciprocal_rank(search_results, expected)

        results.append({
            'query': query,
            'category': category,
            f'precision@{k}': p_at_k,
            'mrr': mrr
        })

    return pd.DataFrame(results)

print("✅ Evaluation functions ready")

# FIXED: Use search_engine.search() directly (no HTML rendering)
def semantic_search_eval(query: str) -> pd.DataFrame:
    results, _ = search_engine.search(query, k=10)
    return results[['name', 'ratings_final', 'price_final', 'similarity']]

# Run Evaluation
print("\n🔄 Running evaluation...\n")

K = 5
semantic_results = evaluate_search_method(semantic_search_eval, test_cases, k=K)
keyword_results = evaluate_search_method(keyword_search, test_cases, k=K)

semantic_results['method'] = 'Semantic Search (FAISS)'
keyword_results['method'] = 'Keyword Search (Baseline)'

comparison_df = pd.concat([semantic_results, keyword_results], ignore_index=True)

print("✅ Evaluation complete!\n")



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("="*80)
print("📊 EVALUATION RESULTS SUMMARY")
print("="*80)

# Overall metrics
for method in ['Semantic Search (FAISS)', 'Keyword Search (Baseline)']:
    method_data = comparison_df[comparison_df['method'] == method]
    avg_precision = method_data[f'precision@{K}'].mean()
    avg_mrr = method_data['mrr'].mean()

    print(f"\n{method}:")
    print(f"  • Average Precision@{K}: {avg_precision:.3f} ({avg_precision*100:.1f}%)")
    print(f"  • Average MRR: {avg_mrr:.3f}")

# Calculate improvement
semantic_p = semantic_results[f'precision@{K}'].mean()
keyword_p = keyword_results[f'precision@{K}'].mean()
improvement = ((semantic_p - keyword_p) / keyword_p) * 100

print(f"\n🚀 Semantic search outperforms keyword search by {improvement:+.1f}%")
print("="*80)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)

# ===== 1. PIVOT TABLE: SIDE-BY-SIDE COMPARISON =====
print("\n" + "="*100)
print("📊 DETAILED COMPARISON TABLE (Side-by-Side)")
print("="*100 + "\n")

# Create pivot for Precision@5
precision_pivot = comparison_df.pivot_table(
    index=['query', 'category'],
    columns='method',
    values='precision@5'
).reset_index()

# Create pivot for MRR
mrr_pivot = comparison_df.pivot_table(
    index=['query', 'category'],
    columns='method',
    values='mrr'
).reset_index()

# Merge both metrics
comparison_table = precision_pivot.merge(
    mrr_pivot,
    on=['query', 'category'],
    suffixes=('_precision', '_mrr')
)

# Rename columns for clarity
comparison_table.columns = [
    'Query', 'Category',
    'Semantic P@5', 'Keyword P@5',
    'Semantic MRR', 'Keyword MRR'
]

# Add delta columns
comparison_table['P@5 Δ'] = comparison_table['Semantic P@5'] - comparison_table['Keyword P@5']
comparison_table['MRR Δ'] = comparison_table['Semantic MRR'] - comparison_table['Keyword MRR']

# Format percentages
for col in ['Semantic P@5', 'Keyword P@5', 'P@5 Δ']:
    comparison_table[col] = comparison_table[col].apply(lambda x: f"{x*100:.0f}%")

for col in ['Semantic MRR', 'Keyword MRR', 'MRR Δ']:
    comparison_table[col] = comparison_table[col].apply(lambda x: f"{x:.3f}")

# Sort by category and delta
comparison_table = comparison_table.sort_values(['Category', 'Query'])

print("\n🔍 Legend: P@5 = Precision@5, MRR = Mean Reciprocal Rank, Δ = Improvement\n")
display(comparison_table)

# ===== 2. BAR CHART: AVERAGE METRICS COMPARISON =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Precision@5 comparison
avg_metrics = comparison_df.groupby('method').agg({
    'precision@5': 'mean',
    'mrr': 'mean'
}).reset_index()

colors = ['#2ecc71', '#e74c3c']
x_pos = np.arange(len(avg_metrics))

# Plot Precision@5
ax1.bar(x_pos, avg_metrics['precision@5'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(avg_metrics['method'], rotation=15, ha='right')
ax1.set_ylabel('Average Precision@5', fontsize=12, fontweight='bold')
ax1.set_title('Average Precision@5 Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(avg_metrics['precision@5']):
    ax1.text(i, v + 0.02, f'{v:.3f}\n({v*100:.1f}%)', ha='center', fontweight='bold', fontsize=11)

# Plot MRR
ax2.bar(x_pos, avg_metrics['mrr'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(avg_metrics['method'], rotation=15, ha='right')
ax2.set_ylabel('Average MRR', fontsize=12, fontweight='bold')
ax2.set_title('Average MRR Comparison', fontsize=14, fontweight='bold')
ax2.set_ylim([0, 1])
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(avg_metrics['mrr']):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

# ===== 3. HEATMAP: PER-QUERY PERFORMANCE =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Precision@5 heatmap
precision_matrix = comparison_df.pivot_table(
    index='query',
    columns='method',
    values='precision@5'
)

sns.heatmap(
    precision_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Precision@5'},
    ax=ax1,
    linewidths=0.5
)
ax1.set_title('Precision@5 Heatmap (per Query)', fontsize=14, fontweight='bold')
ax1.set_xlabel('')
ax1.set_ylabel('Query', fontsize=11)

# MRR heatmap
mrr_matrix = comparison_df.pivot_table(
    index='query',
    columns='method',
    values='mrr'
)

sns.heatmap(
    mrr_matrix,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'MRR'},
    ax=ax2,
    linewidths=0.5
)
ax2.set_title('MRR Heatmap (per Query)', fontsize=14, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

# ===== 4. CATEGORY-WISE COMPARISON =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

category_metrics = comparison_df.groupby(['category', 'method']).agg({
    'precision@5': 'mean',
    'mrr': 'mean'
}).reset_index()

categories = category_metrics['category'].unique()
x = np.arange(len(categories))
width = 0.35

# Precision@5 by category
semantic_p5 = category_metrics[category_metrics['method'] == 'Semantic Search (FAISS)']['precision@5'].values
keyword_p5 = category_metrics[category_metrics['method'] == 'Keyword Search (Baseline)']['precision@5'].values

ax1.bar(x - width/2, semantic_p5, width, label='Semantic', color='#2ecc71', alpha=0.8, edgecolor='black')
ax1.bar(x + width/2, keyword_p5, width, label='Keyword', color='#e74c3c', alpha=0.8, edgecolor='black')
ax1.set_xlabel('Category', fontsize=12, fontweight='bold')
ax1.set_ylabel('Average Precision@5', fontsize=12, fontweight='bold')
ax1.set_title('Precision@5 by Category', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(categories, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1.1])

# MRR by category
semantic_mrr = category_metrics[category_metrics['method'] == 'Semantic Search (FAISS)']['mrr'].values
keyword_mrr = category_metrics[category_metrics['method'] == 'Keyword Search (Baseline)']['mrr'].values

ax2.bar(x - width/2, semantic_mrr, width, label='Semantic', color='#2ecc71', alpha=0.8, edgecolor='black')
ax2.bar(x + width/2, keyword_mrr, width, label='Keyword', color='#e74c3c', alpha=0.8, edgecolor='black')
ax2.set_xlabel('Category', fontsize=12, fontweight='bold')
ax2.set_ylabel('Average MRR', fontsize=12, fontweight='bold')
ax2.set_title('MRR by Category', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

# ===== 5. WIN/LOSS SUMMARY =====
print("\n" + "="*100)
print("🏆 WIN/LOSS SUMMARY")
print("="*100 + "\n")

# Count wins per metric
precision_wins = (precision_matrix['Semantic Search (FAISS)'] > precision_matrix['Keyword Search (Baseline)']).sum()
mrr_wins = (mrr_matrix['Semantic Search (FAISS)'] > mrr_matrix['Keyword Search (Baseline)']).sum()

total_queries = len(test_cases)

print(f"📊 Out of {total_queries} test queries:\n")
print(f"  🟢 Semantic wins (Precision@5): {precision_wins}/{total_queries} ({precision_wins/total_queries*100:.0f}%)")
print(f"  🟢 Semantic wins (MRR):          {mrr_wins}/{total_queries} ({mrr_wins/total_queries*100:.0f}%)")
print(f"\n  🔴 Keyword wins (Precision@5):   {total_queries - precision_wins}/{total_queries} ({(total_queries - precision_wins)/total_queries*100:.0f}%)")
print(f"  🔴 Keyword wins (MRR):           {total_queries - mrr_wins}/{total_queries} ({(total_queries - mrr_wins)/total_queries*100:.0f}%)")
print("\n" + "="*100)

In [ ]:
from itertools import product
cache_results = {}
for case in test_cases:
    res, _ = search_engine.search(case['query'], k=20)
    res['ratings_norm'] = (res['ratings_final'].astype(float)-1)/4
    p_min, p_max = res['price_final'].min(), res['price_final'].max()
    res['price_norm'] = 1 - (res['price_final'] - p_min) / (p_max - p_min + 1e-6)
    cache_results[case['query']] = res

#grid search
weight_candidates = [
    (ws, wr, wp)
    for ws in [0.5, 0.6, 0.7, 0.8]
    for wr in [0.1, 0.2, 0.3]
    for wp in [0.05, 0.1, 0.15, 0.2]
    if abs((ws + wr + wp) - 1.0) < 1e-6
]
best_weights = None
best_mrr = 0

for ws, wr, wp in weight_candidates:
    total_mrr = 0
    for case in test_cases:
        res = cache_results[case['query']].copy()
        # Re-rank based on new weights
        res['final_score'] = (res['similarity'] * ws +
                             res['ratings_norm'] * wr +
                             res['price_norm'] * wp)

        sorted_res = res.sort_values('final_score', ascending=False)
        total_mrr += mean_reciprocal_rank(sorted_res, case['expected_keywords'])
    avg_mrr = total_mrr/len(test_cases)
    if avg_mrr > best_mrr:
        best_mrr = avg_mrr
        best_weights = (ws, wr, wp)
print(f"--- OPTIMIZATION RESULT ---")
print(f"Best weights: Similarity={best_weights[0]}, Rating={best_weights[1]}, Price={best_weights[2]}")
print(f"Best MRR: {best_mrr:.4f}")

In [ ]:


if 'search_engine' not in globals() or not search_engine._initialized:
    raise RuntimeError("❌ Chạy cell SearchEngine trước!")

import ipywidgets as widgets
from IPython.display import display, clear_output

search_input = widgets.Text(
    placeholder='e.g. gaming laptop, cheap earphones, yoga mat...',
    layout=widgets.Layout(width='550px', height='38px')
)
search_button = widgets.Button(
    description='🔍 Search',
    button_style='primary',
    layout=widgets.Layout(width='110px', height='38px')
)
top_k_slider = widgets.IntSlider(
    value=10, min=5, max=20, step=1,
    description='Results:',
    layout=widgets.Layout(width='380px')
)
output_area = widgets.Output()


def run_search(query: str, top_k: int):
    if not query.strip():
        return
    ws, wr, wp = best_weights
    results, meta = search_engine.search(query, k=top_k)
    results = results.copy()
    results['rating_norm'] = (results['ratings_final'].astype(float) - 1) / 4
    p_min, p_max = results['price_final'].min(), results['price_final'].max()
    results['price_norm'] = 1 - (results['price_final'] - p_min) / (p_max - p_min + 1e-6)
    results['final_score'] = (
        results['similarity']   * ws +
        results['rating_norm']  * wr +
        results['price_norm']   * wp
    )
    results = results.sort_values('final_score', ascending=False).reset_index(drop=True)

    cards_html = ""
    for i, row in results.head(top_k).iterrows():
        rank         = i + 1
        rank_icon    = {1: "🥇", 2: "🥈", 3: "🥉"}.get(rank, f"#{rank}")
        full_name    = str(row['name'])
        short_name   = full_name[:100] + "…" if len(full_name) > 100 else full_name
        is_long      = len(full_name) > 100

        sim_pct      = float(row['similarity'])  * 100
        rat_pct      = float(row['rating_norm']) * 100
        pri_pct      = float(row['price_norm'])  * 100
        score        = float(row['final_score'])
        cat          = str(row.get('main_category') or '—')
        sub          = str(row.get('sub_category')  or '—')
        price        = float(row['price_final'])
        rating       = row['ratings_final']

        border_color = {1: "#FFD700", 2: "#A0A0A0", 3: "#CD7F32"}.get(rank, "#3B82F6")
        card_id      = f"card_detail_{rank}"

        # Score breakdown dạng công thức
        sim_contrib  = float(row['similarity'])  * ws
        rat_contrib  = float(row['rating_norm']) * wr
        pri_contrib  = float(row['price_norm'])  * wp

        # Toggle button chỉ hiện nếu tên dài
        toggle_btn = f"""
            <span
                onclick="
                    var el = document.getElementById('{card_id}');
                    var btn = document.getElementById('btn_{card_id}');
                    if(el.style.display==='none'){{
                        el.style.display='block';
                        btn.innerText='▲ Thu gọn';
                    }} else {{
                        el.style.display='none';
                        btn.innerText='▼ Xem thêm';
                    }}
                "
                id="btn_{card_id}"
                style="
                    cursor:pointer;
                    font-size:11px;
                    color:#3B82F6;
                    font-weight:600;
                    margin-left:6px;
                    user-select:none;
                ">▼ Xem thêm</span>
        """ if is_long else ""

        full_name_block = f"""
            <div id="{card_id}" style="display:none; margin-top:6px;
                background:#F9FAFB; border-radius:8px; padding:10px 14px;
                font-size:13px; color:#374151; line-height:1.6;">
                {full_name}
            </div>
        """ if is_long else ""

        cards_html += f"""
        <div style="
            background:white;
            border-left:6px solid {border_color};
            border-radius:14px;
            padding:18px 22px;
            margin-bottom:14px;
            box-shadow:0 3px 12px rgba(0,0,0,0.07);
            font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
        ">
            <!-- Rank + tên ngắn -->
            <div style="display:flex;align-items:flex-start;gap:10px;margin-bottom:10px;">
                <span style="
                    font-size:20px;font-weight:800;
                    background:{border_color}22;border:1px solid {border_color}55;
                    border-radius:8px;padding:3px 10px;color:{border_color};
                    white-space:nowrap;
                ">{rank_icon}</span>
                <div style="flex:1;">
                    <span style="font-size:15px;font-weight:700;color:#111827;line-height:1.4;">
                        {short_name}
                    </span>
                    {toggle_btn}
                    {full_name_block}
                </div>
            </div>

            <!-- Meta chips -->
            <div style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:14px;">
                <span style="background:#FEF3C7;color:#92400E;border-radius:6px;padding:3px 10px;font-size:12px;font-weight:600;">⭐ {rating}</span>
                <span style="background:#ECFDF5;color:#065F46;border-radius:6px;padding:3px 10px;font-size:12px;font-weight:600;">₹{price:,.0f}</span>
                <span style="background:#EFF6FF;color:#1D4ED8;border-radius:6px;padding:3px 10px;font-size:12px;font-weight:600;">📂 {cat}</span>
                <span style="background:#F5F3FF;color:#5B21B6;border-radius:6px;padding:3px 10px;font-size:12px;font-weight:600;">🏷 {sub}</span>
                <span style="background:#F0FDF4;color:#166534;border-radius:6px;padding:3px 10px;font-size:12px;font-weight:700;">Score: {score:.3f}</span>
            </div>

            <!-- Score bars -->
            <div style="margin-bottom:10px;">
                {''.join([
                    f"""<div style="margin-bottom:7px;">
                        <div style="display:flex;justify-content:space-between;margin-bottom:3px;">
                            <span style="font-size:11px;font-weight:700;color:#6B7280;">{label}</span>
                            <span style="font-size:11px;font-weight:700;color:{color};">{pct:.1f}%</span>
                        </div>
                        <div style="background:#E5E7EB;border-radius:4px;height:6px;overflow:hidden;">
                            <div style="width:{min(pct,100)}%;height:100%;background:{color};border-radius:4px;"></div>
                        </div>
                    </div>"""
                    for label, pct, color in [
                        ("🎯 SIMILARITY",     sim_pct, "linear-gradient(90deg,#3B82F6,#6366F1)"),
                        ("⭐ RATING QUALITY", rat_pct, "linear-gradient(90deg,#10B981,#059669)"),
                        ("💰 PRICE VALUE",    pri_pct, "linear-gradient(90deg,#F59E0B,#D97706)"),
                    ]
                ])}
            </div>

            <!-- Score formula - click để toggle -->
            <div
                onclick="
                    var el=document.getElementById('formula_{rank}');
                    el.style.display=el.style.display==='none'?'block':'none';
                "
                style="
                    cursor:pointer;
                    font-size:11px;color:#9CA3AF;
                    font-weight:600;text-align:right;
                    user-select:none;
                ">
                🧮 Xem công thức tính score ▾
            </div>
            <div id="formula_{rank}" style="display:none;
                margin-top:8px;background:#F9FAFB;border-radius:10px;
                padding:12px 16px;font-size:12px;color:#374151;line-height:2;
                border:1px solid #E5E7EB;">
                <b>Final Score = (Similarity × 0.7) + (Rating × 0.2) + (Price × 0.1)</b><br>
                &nbsp;&nbsp;&nbsp;= ({sim_pct/100:.3f} × 0.7) &nbsp;+&nbsp; ({rat_pct/100:.3f} × 0.2) &nbsp;+&nbsp; ({pri_pct/100:.3f} × 0.1)<br>
                &nbsp;&nbsp;&nbsp;= {sim_contrib:.3f} &nbsp;+&nbsp; {rat_contrib:.3f} &nbsp;+&nbsp; {pri_contrib:.3f}<br>
                &nbsp;&nbsp;&nbsp;= <b style="color:#3B82F6;">{score:.3f}</b>
            </div>
        </div>
        """

    full_html = f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;max-width:860px;">
        <div style="
            background:linear-gradient(135deg,#EFF6FF,#F5F3FF);
            border:1px solid #DBEAFE;border-radius:12px;
            padding:12px 20px;margin-bottom:20px;
            display:flex;gap:24px;align-items:center;flex-wrap:wrap;
        ">
            <span style="font-size:13px;color:#6B7280;">⚡ <b style="color:#111827;">{meta['latency_ms']:.1f}ms</b></span>
            <span style="font-size:13px;color:#6B7280;">📦 <b style="color:#111827;">{meta['index_size']:,}</b> products</span>
            <span style="font-size:13px;color:#6B7280;">🎯 Top <b style="color:#111827;">{top_k}</b> for <b style="color:#3B82F6;">"{query}"</b></span>
        </div>
        {cards_html}
    </div>
    """

    with output_area:
        clear_output(wait=True)
        displayHTML(full_html)


search_button.on_click(lambda b: run_search(search_input.value, top_k_slider.value))

display(widgets.VBox([
    widgets.HBox([search_input, search_button]),
    top_k_slider,
    output_area
]))